# GR00T-N1.7 HoloQ W4A4 — Modal L4 Phase 1

Notebook này chạy **một suite LIBERO tại một thời điểm** trên Modal L4. Gắn một Modal Volume tại `/vol`, chọn kernel **NVIDIA L4** (khuyến nghị ít nhất 8 CPU / 64 GiB RAM), đặt `HF_TOKEN` bằng Modal Secret nếu model yêu cầu xác thực, rồi chỉ thay `SUITE` và `WORK_PHASE` trong cell cấu hình và chọn **Run all**.

Thứ tự bắt buộc: `setup → prepare → calibrate → build_pack → smoke_rollout → full_rollout → package`. Phase `status` chỉ đọc tiến độ. `full_rollout` chạy cả BF16 và W4A4 bằng cùng seed; mỗi task được ghi atomically nên chạy lại sẽ resume. `package` tạo ZIP theo suite để tải xuống và upload thành Kaggle Dataset.

`setup` chỉ clone `duc-quan` bằng `--single-branch`; notebook không fetch hoặc checkout `main` hay branch khác. Checkpoint Hugging Face được resolve thành commit SHA bất biến trong `prepare`.

In [ ]:
# ============================================================
# CHỈNH HAI BIẾN NÀY, SAU ĐÓ CHỌN RUN ALL
# ============================================================
SUITE = "object"
WORK_PHASE = "setup"

# SUITE: object | spatial | goal | long
# WORK_PHASE:
#   setup | prepare | calibrate | build_pack | smoke_rollout
#   full_rollout | package | status

# Reproducibility and storage
REPO_URL = "https://github.com/hungho77/Isaac-GR00T.git"
SOURCE_BRANCH = "duc-quan"
VOLUME_ROOT = "/vol/gr00t-n17-holoq"
CHECKPOINT_REPO = "nvidia/GR00T-N1.7-LIBERO"
CHECKPOINT_REF = ""  # empty => resolve current main to an immutable SHA
FORCE_SETUP = False

# Evaluation protocol
SERVER_PORT = 5555
CALIBRATION_SEED = 0
EVALUATION_SEED_BASE = 10000
N_ENVS = 1
N_ACTION_STEPS = 8
MAX_EPISODE_STEPS = 720
CALIBRATION_TOPK = 512
SMOKE_TASK_INDEX = 0

# Video được tạo bởi rollout wrapper; False sẽ xóa sau khi JSON đã ghi xong.
RECORD_VIDEOS = False
PACKAGE_INCLUDE_VIDEOS = False
REQUIRE_L4 = True


In [ ]:
# Re-entrant patch: make Git ignore only LIBERO setup-generated residue.
# This runs before bootstrap/prepare and never resets, cleans, or deletes files.
from pathlib import Path as _PatchPath
import subprocess as _patch_subprocess

_patch_repo = _PatchPath(VOLUME_ROOT).resolve() / "Isaac-GR00T-duc-quan"
if (_patch_repo / ".git").is_dir():
    _patch_subprocess.run(
        [
            "git", "-C", str(_patch_repo), "config",
            "submodule.external_dependencies/LIBERO.ignore", "dirty",
        ],
        check=True,
    )
    _exclude = _patch_repo / ".git" / "info" / "exclude"
    _exclude.parent.mkdir(parents=True, exist_ok=True)
    _rule = "/gr00t/eval/sim/LIBERO/libero_uv/"
    _existing = _exclude.read_text(encoding="utf-8") if _exclude.is_file() else ""
    if _rule not in _existing.splitlines():
        with _exclude.open("a", encoding="utf-8") as _stream:
            if _existing and not _existing.endswith("\n"):
                _stream.write("\n")
            _stream.write(_rule + "\n")
    _remaining = _patch_subprocess.check_output(
        ["git", "-C", str(_patch_repo), "status", "--porcelain"],
        text=True,
    ).strip()
    if _remaining:
        raise RuntimeError(
            "Repository still has real source changes after the safe residue patch:\n"
            + _remaining
        )
    print("GIT RESIDUE PATCH OK: repository is reproducibly clean")
else:
    print("GIT RESIDUE PATCH: repo not cloned yet; setup will create it")


## Bootstrap và phase dispatcher

Cell dưới đây không cần chỉnh. `setup` lưu repo, môi trường và artifact trên Volume; các phase sau gọi runner được khóa trong chính repo. LIBERO luôn dùng path mặc định của submodule, không hỏi Y/N; nếu setup trước bị dừng sau khi tạo venv, cell sẽ kiểm tra và tái sử dụng venv trước khi dựng lại.

In [ ]:
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile

from IPython.display import FileLink, display

VALID_SUITES = {"object", "spatial", "goal", "long"}
VALID_PHASES = {
    "setup", "prepare", "calibrate", "build_pack",
    "smoke_rollout", "full_rollout", "package", "status",
}
if SUITE not in VALID_SUITES:
    raise ValueError(f"SUITE must be one of {sorted(VALID_SUITES)}; got {SUITE!r}")
if WORK_PHASE not in VALID_PHASES:
    raise ValueError(f"WORK_PHASE must be one of {sorted(VALID_PHASES)}; got {WORK_PHASE!r}")
if SOURCE_BRANCH != "duc-quan":
    raise ValueError("This notebook is intentionally locked to branch 'duc-quan'")

volume_root = Path(VOLUME_ROOT).resolve()
volume_mount = Path("/vol")
repo_path = volume_root / "Isaac-GR00T-duc-quan"
artifact_root = volume_root / "artifacts"
setup_marker = volume_root / "setup.json"

def run(command, *, cwd=None, env=None, capture_output=False):
    command = [str(value) for value in command]
    print("+", " ".join(command))
    return subprocess.run(
        command, cwd=cwd, env=env, check=True, text=True,
        capture_output=capture_output,
    )

def git(*arguments, capture_output=False):
    return run(
        ["git", "-C", repo_path, *arguments],
        capture_output=capture_output,
    )

def restore_libero_config():
    persisted = artifact_root / "libero_home_config"
    target = Path.home() / ".libero"
    if persisted.is_dir() and not target.exists():
        shutil.copytree(persisted, target)

def write_libero_config():
    libero_root = repo_path / "external_dependencies/LIBERO/libero/libero"
    if not libero_root.is_dir():
        raise RuntimeError("LIBERO submodule is missing; initialize it before configuration")
    config_dir = Path.home() / ".libero"
    config_dir.mkdir(parents=True, exist_ok=True)
    config_path = config_dir / "config.yaml"
    values = {
        "benchmark_root": libero_root,
        "bddl_files": libero_root / "bddl_files",
        "init_states": libero_root / "init_files",
        "datasets": libero_root.parent / "datasets",
        "assets": libero_root / "assets",
    }
    config_path.write_text(
        "".join(
            f"{key}: {Path(value).resolve().as_posix()}\n"
            for key, value in values.items()
        ),
        encoding="utf-8",
    )
    print(f"Wrote non-interactive LIBERO config: {config_path}")
    return config_dir

def blocking_repo_changes():
    status = git(
        "status", "--porcelain", "--untracked-files=all",
        "--ignore-submodules=dirty", capture_output=True,
    ).stdout.splitlines()
    generated_prefix = "gr00t/eval/sim/LIBERO/libero_uv/"
    ignored, blocking = [], []
    for line in status:
        path = line[3:] if len(line) > 3 else line
        (ignored if path.startswith(generated_prefix) else blocking).append(line)
    if ignored:
        print("Ignoring generated LIBERO setup residue:", *ignored, sep="\n  " )
    return blocking

def libero_environment():
    environment = os.environ.copy()
    environment.update({
        "MPLBACKEND": "Agg",
        "MUJOCO_GL": "egl",
        "PYOPENGL_PLATFORM": "egl",
        "PYTHONUNBUFFERED": "1",
    })
    return environment

LIBERO_SMOKE_CODE = """
from gr00t.eval.sim.LIBERO.libero_env import register_libero_envs
register_libero_envs()
import gymnasium as gym
env = gym.make("libero_sim/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate")
env.reset()
env.close()
print("LIBERO env OK:", type(env))
"""

def persist_completed_setup(source_revision, *, recovery=None):
    live_config = Path.home() / ".libero"
    if not live_config.is_dir():
        raise RuntimeError("LIBERO setup did not create ~/.libero")
    persisted_config = artifact_root / "libero_home_config"
    shutil.copytree(live_config, persisted_config, dirs_exist_ok=True)
    marker = {
        "source_branch": SOURCE_BRANCH,
        "source_revision": source_revision,
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    if recovery:
        marker["recovery"] = recovery
    setup_marker.write_text(json.dumps(marker, indent=2) + "\n")

def try_resume_partial_setup(source_revision):
    libero_python = repo_path / "gr00t/eval/sim/LIBERO/libero_uv/.venv/bin/python"
    if setup_marker.is_file() or not libero_python.is_file():
        return False
    print("Partial LIBERO venv found; validating it without rebuilding...")
    try:
        write_libero_config()
        run(
            [libero_python, "-c", LIBERO_SMOKE_CODE],
            cwd=repo_path, env=libero_environment(),
        )
    except (OSError, subprocess.CalledProcessError, RuntimeError) as error:
        print(f"Partial validation failed ({error}); falling back to full setup")
        return False
    persist_completed_setup(
        source_revision, recovery="noninteractive_partial_setup",
    )
    print("PARTIAL SETUP RECOVERED: existing venv reused")
    return True

def run_noninteractive_libero_setup(environment):
    source = repo_path / "gr00t/eval/sim/LIBERO/setup_libero.sh"
    script = source.read_text(encoding="utf-8")
    lines = script.splitlines(keepends=True)
    prompt_lines = [
        index for index, line in enumerate(lines)
        if "register_libero_envs" in line and "|" in line
    ]
    noninteractive = """python - <<PY
from pathlib import Path
root = (Path("$LIBERO_REPO") / "libero" / "libero").resolve()
config = Path.home() / ".libero" / "config.yaml"
config.parent.mkdir(parents=True, exist_ok=True)
values = {
    "benchmark_root": root,
    "bddl_files": root / "bddl_files",
    "init_states": root / "init_files",
    "datasets": root.parent / "datasets",
    "assets": root / "assets",
}
config.write_text("".join(f"{k}: {Path(v).as_posix()}\\n" for k, v in values.items()))
print(f"Wrote non-interactive LIBERO config: {config}")
PY
python -c "from gr00t.eval.sim.LIBERO.libero_env import register_libero_envs; register_libero_envs()"
"""
    if prompt_lines:
        prompt_index = prompt_lines[0]
        start_index = prompt_index
        if prompt_index and ".libero" in lines[prompt_index - 1]:
            start_index -= 1
        lines[start_index : prompt_index + 1] = [noninteractive]
        script = "".join(lines)
    elif "Wrote non-interactive LIBERO config" not in script:
        raise RuntimeError("Unknown LIBERO setup script; refusing to run an interactive variant")
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".sh", prefix=".modal-noninteractive-",
        dir=source.parent, encoding="utf-8", delete=False,
    ) as stream:
        stream.write(script)
        temporary_script = Path(stream.name)
    try:
        run(["bash", temporary_script], cwd=repo_path, env=environment)
    finally:
        temporary_script.unlink(missing_ok=True)

def bootstrap_setup():
    if not volume_mount.is_dir():
        raise RuntimeError("Attach a Modal Volume at /vol before running setup")
    volume_root.mkdir(parents=True, exist_ok=True)
    artifact_root.mkdir(parents=True, exist_ok=True)

    run(["apt-get", "update"])
    run([
        "apt-get", "install", "-y", "--no-install-recommends",
        "git", "git-lfs", "ffmpeg", "libegl1-mesa-dev",
        "libglu1-mesa", "libgl1", "libglib2.0-0",
    ])
    if shutil.which("uv") is None:
        run([sys.executable, "-m", "pip", "install", "uv"])

    if not (repo_path / ".git").is_dir():
        run([
            "git", "clone", "--single-branch", "--branch", SOURCE_BRANCH,
            REPO_URL, repo_path,
        ])
    else:
        blocking = blocking_repo_changes()
        if blocking:
            raise RuntimeError(
                "Persistent repo has source changes; refusing to overwrite:\n"
                + "\n".join(blocking)
            )
        if FORCE_SETUP:
            git("fetch", "origin", SOURCE_BRANCH)
            git("checkout", "--detach", "FETCH_HEAD")

    source_revision = git("rev-parse", "HEAD", capture_output=True).stdout.strip()
    if not FORCE_SETUP:
        try_resume_partial_setup(source_revision)
    previous = json.loads(setup_marker.read_text()) if setup_marker.is_file() else {}
    needs_install = FORCE_SETUP or previous.get("source_revision") != source_revision
    if needs_install:
        git("submodule", "update", "--init", "external_dependencies/LIBERO")
        environment = os.environ.copy()
        environment["UV_LINK_MODE"] = "copy"
        environment["UV_CACHE_DIR"] = str(volume_root / "uv-cache")
        environment["MPLBACKEND"] = "Agg"
        environment["MUJOCO_GL"] = "egl"
        environment["PYOPENGL_PLATFORM"] = "egl"
        run(["uv", "sync", "--locked"], cwd=repo_path, env=environment)
        run_noninteractive_libero_setup(environment)
        persist_completed_setup(source_revision)
    else:
        print(f"Setup already complete for {source_revision}; set FORCE_SETUP=True to rebuild")
    restore_libero_config()
    print(f"SETUP COMPLETE: repo={repo_path}, revision={source_revision}")

def dispatch_phase():
    if not (repo_path / ".git").is_dir():
        raise RuntimeError("Persistent repo is missing; run WORK_PHASE='setup' first")
    restore_libero_config()
    runner = repo_path / "examples/LIBERO/quantization/modal_phase1_runner.py"
    python = repo_path / ".venv/bin/python"
    if not runner.is_file() or not python.is_file():
        raise RuntimeError("Phase runner or server environment is missing; rerun setup")
    command = [
        python, runner,
        "--suite", SUITE,
        "--work-phase", WORK_PHASE,
        "--repo-path", repo_path,
        "--artifact-root", artifact_root,
        "--checkpoint-repo", CHECKPOINT_REPO,
        "--checkpoint-ref", CHECKPOINT_REF,
        "--server-port", SERVER_PORT,
        "--calibration-seed", CALIBRATION_SEED,
        "--evaluation-seed-base", EVALUATION_SEED_BASE,
        "--n-envs", N_ENVS,
        "--n-action-steps", N_ACTION_STEPS,
        "--max-episode-steps", MAX_EPISODE_STEPS,
        "--calibration-topk", CALIBRATION_TOPK,
        "--smoke-task-index", SMOKE_TASK_INDEX,
        "--record-videos" if RECORD_VIDEOS else "--no-record-videos",
        "--package-include-videos" if PACKAGE_INCLUDE_VIDEOS else "--no-package-include-videos",
        "--require-l4" if REQUIRE_L4 else "--no-require-l4",
    ]
    environment = os.environ.copy()
    environment.update({
        "MPLBACKEND": "Agg",
        "MUJOCO_GL": "egl",
        "PYOPENGL_PLATFORM": "egl",
        "UV_CACHE_DIR": str(volume_root / "uv-cache"),
        "HF_HOME": str(volume_root / "hf-cache"),
    })
    run(command, cwd=repo_path, env=environment)

if WORK_PHASE == "setup":
    bootstrap_setup()
else:
    dispatch_phase()


## Kết quả package

Khi `WORK_PHASE="package"`, cell cuối hiển thị link tải ZIP và file SHA256. Với phase khác, cell chỉ in vị trí artifact hiện tại.

In [ ]:
last_export_path = artifact_root / "last_export.json"
if WORK_PHASE == "package":
    if not last_export_path.is_file():
        raise RuntimeError("Package phase finished without last_export.json")
    export = json.loads(last_export_path.read_text())
    if export.get("suite") != SUITE:
        raise RuntimeError(f"Last export suite mismatch: {export.get('suite')} != {SUITE}")
    print(json.dumps(export, indent=2, ensure_ascii=False))
    display(FileLink(export["zip_path"]))
    display(FileLink(export["sha256_path"]))
else:
    print(f"Phase {WORK_PHASE!r} complete for suite {SUITE!r}. Artifacts: {artifact_root}")
